# Pregunta 6: Umbral equivalente de wet days (CR2MET vs ALADIN)

**Objetivo:** encontrar el umbral diario en **ALADIN** (mm/día) para que la **fracción de wet days integrada en el dominio** coincida con la de **CR2MET**, cuando CR2MET usa **0,1 mm** o **1,0 mm** como referencia fija.

**Referencia metodológica:** sección **3.b** de Martinez-Villalobos et al. (2022), [JCLI-D-21-0590](https://journals.ametsoc.org/view/journals/clim/35/12/JCLI-D-21-0590.1.xml) (*Extremes: daily precipitation and duration of dry spells*), y material suplementario ([SI](https://doi.org/10.1175/JCLI-D-21-0590.s1)): umbral **dependiente del modelo** que iguala la frecuencia de días húmedos al producto de referencia antes de comparar métricas de precipitación / dry spells.

**Dominio y datos** (igual que `pruea.ipynb`):
- Chile continental (Natural Earth) sobre grilla **ALADIN CHP12**
- **CR2MET** interpolado linealmente a esa grilla
- Periodo histórico común **1980–2014**
- Fracción integrada = **promedio espacial** sobre celdas válidas del % temporal `(pr ≥ τ)`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep
import cartopy.io.shapereader as shpreader

plt.rcParams['figure.figsize'] = (10, 6)

# =====================================================================
# CONFIGURACION
# =====================================================================
START_DATE = '1980-01-01'
END_DATE = '2014-12-31'
CR2MET_REFERENCE_THRESHOLDS = [0.1, 1.0]  # mm/dia
THRESHOLD_GRID = np.linspace(0.0, 8.0, 161)
BISECTION_TAU_MAX = 15.0
BISECTION_TOL = 1e-4  # mm/dia


def load_chile_geometry():
    reader = shpreader.Reader(
        shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
    )
    geoms = [
        rec.geometry
        for rec in reader.records()
        if rec.attributes.get('NAME') == 'Chile' or rec.attributes.get('ADMIN') == 'Chile'
    ]
    if not geoms:
        raise RuntimeError('No se encontro Chile en Natural Earth')
    return unary_union(geoms)


def build_chile_mask_on_aladin_grid(lat2d, lon2d, geometry):
    prepared = prep(geometry)
    flat_mask = np.fromiter(
        (
            prepared.contains(Point(float(x), float(y))) or geometry.touches(Point(float(x), float(y)))
            for y, x in zip(lat2d.ravel(), lon2d.ravel())
        ),
        dtype=bool,
        count=lat2d.size,
    )
    return flat_mask.reshape(lat2d.shape)


def open_aladin_historical():
    files = sorted(Path('pr1').glob('pr_CHP12_*_historical_*.nc'))
    if not files:
        raise FileNotFoundError('No hay ALADIN historico en ./pr1/')
    ds = xr.open_mfdataset([str(p) for p in files], use_cftime=True, chunks={'time': 365})
    return ds['pr'].sel(time=slice(START_DATE, END_DATE)) * 86400.0


def open_cr2met_historical():
    files = sorted(Path('pr').glob('CR2MET_pr_v2.5_day_*.nc'))
    if not files:
        raise FileNotFoundError('No hay CR2MET en ./pr/')
    ds = xr.open_mfdataset([str(p) for p in files], combine='by_coords', chunks={'time': 365})
    return ds['pr'].sel(time=slice(START_DATE, END_DATE))


def domain_mean_wet_fraction(pr, threshold, mask_da):
    """Promedio espacial del % temporal de dias con pr >= threshold (celdas validas)."""
    frac = (pr >= threshold).mean(dim='time') * 100.0
    vals = frac.where(mask_da).values.ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        raise RuntimeError('Sin celdas validas para calcular fraccion wet day')
    return float(np.mean(vals))


def wet_fraction_curve(pr, thresholds, mask_da):
    return np.array([domain_mean_wet_fraction(pr, float(t), mask_da) for t in thresholds])


def find_aladin_threshold_for_target(pr_aladin, target_fraction, mask_da, tau_max=BISECTION_TAU_MAX, tol=BISECTION_TOL):
    """Busca tau* tal que F_ALADIN(tau*) = target_fraction (F decrece con tau)."""
    f0 = domain_mean_wet_fraction(pr_aladin, 0.0, mask_da)
    fmax = domain_mean_wet_fraction(pr_aladin, tau_max, mask_da)

    if target_fraction > f0 + tol:
        raise ValueError(f'Target {target_fraction:.3f}% > F_ALADIN(0)={f0:.3f}%')
    if target_fraction < fmax - tol:
        raise ValueError(
            f'Target {target_fraction:.3f}% < F_ALADIN({tau_max})={fmax:.3f}%; aumenta tau_max'
        )

    lo, hi = 0.0, tau_max
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        fmid = domain_mean_wet_fraction(pr_aladin, mid, mask_da)
        if abs(fmid - target_fraction) <= tol:
            return mid, fmid
        if fmid > target_fraction:
            lo = mid
        else:
            hi = mid
    tau_star = 0.5 * (lo + hi)
    return tau_star, domain_mean_wet_fraction(pr_aladin, tau_star, mask_da)


print('1/3: Cargando CR2MET y ALADIN en dominio comun...')
pr_aladin = open_aladin_historical()
pr_cr2met_native = open_cr2met_historical()

chile_geom = load_chile_geometry()
chile_mask_bool = build_chile_mask_on_aladin_grid(
    pr_aladin['lat'].values,
    pr_aladin['lon'].values,
    chile_geom,
)
chile_mask = xr.DataArray(
    chile_mask_bool,
    coords={'y': pr_aladin['y'], 'x': pr_aladin['x'], 'lat': pr_aladin['lat'], 'lon': pr_aladin['lon']},
    dims=['y', 'x'],
    name='chile_mask',
)

pr_cr2met = pr_cr2met_native.interp(
    lat=pr_aladin['lat'],
    lon=pr_aladin['lon'],
    method='linear',
)
pr_cr2met_chile = pr_cr2met.where(chile_mask)
pr_aladin_chile = pr_aladin.where(chile_mask)

print(f'Periodo: {START_DATE} a {END_DATE}')
print(f'Celdas validas Chile (grilla ALADIN): {int(chile_mask.sum().values)}')
print('ALADIN: kg m-2 s-1 -> mm/dia (x86400). CR2MET: mm/dia.')

In [ ]:
# =====================================================================
# UMBRAL EQUIVALENTE ALADIN (DOS REFERENCIAS CR2MET)
# =====================================================================
print('2/3: Calculando umbrales equivalentes (puede tardar unos minutos)...')

curve_aladin = wet_fraction_curve(pr_aladin_chile, THRESHOLD_GRID, chile_mask)

rows = []
for tau_cr2 in CR2MET_REFERENCE_THRESHOLDS:
    f_cr2 = domain_mean_wet_fraction(pr_cr2met_chile, tau_cr2, chile_mask)
    f_aladin_fixed = domain_mean_wet_fraction(pr_aladin_chile, tau_cr2, chile_mask)
    tau_aladin_star, f_aladin_star = find_aladin_threshold_for_target(
        pr_aladin_chile, f_cr2, chile_mask
    )
    rows.append({
        'tau_CR2MET (mm/dia)': tau_cr2,
        'F_CR2MET integrada (%)': f_cr2,
        'F_ALADIN con mismo tau (%)': f_aladin_fixed,
        'Sesgo ALADIN sin ajuste (pp)': f_aladin_fixed - f_cr2,
        'tau_ALADIN equivalente (mm/dia)': tau_aladin_star,
        'F_ALADIN(tau*) verificacion (%)': f_aladin_star,
        'Error matching (pp)': f_aladin_star - f_cr2,
    })

results_df = pd.DataFrame(rows)
display(results_df.round(4))

print('\n--- Lectura ---')
for _, r in results_df.iterrows():
    print(
        f"Si CR2MET usa tau={r['tau_CR2MET (mm/dia)']:.1f} mm/dia "
        f"(F={r['F_CR2MET integrada (%)']:.2f}%), "
        f"ALADIN necesita tau*={r['tau_ALADIN equivalente (mm/dia)']:.3f} mm/dia "
        f"para igualar la fraccion integrada de wet days."
    )

In [ ]:
# =====================================================================
# FIGURA: F(tau) INTEGRADA EN EL DOMINIO
# =====================================================================
print('3/3: Graficando curvas F(tau)...')

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(THRESHOLD_GRID, curve_aladin, color='steelblue', linewidth=2.2, label='ALADIN (curva F(tau))')

colors = {0.1: 'darkorange', 1.0: 'firebrick'}
for _, r in results_df.iterrows():
    tau_cr2 = r['tau_CR2MET (mm/dia)']
    f_cr2 = r['F_CR2MET integrada (%)']
    tau_star = r['tau_ALADIN equivalente (mm/dia)']
    color = colors.get(tau_cr2, 'black')

    ax.scatter([tau_cr2], [f_cr2], s=90, color=color, zorder=5,
               label=f'CR2MET @ {tau_cr2:g} mm/dia (F={f_cr2:.1f}%)')
    ax.scatter([tau_star], [f_cr2], s=90, facecolors='white', edgecolors=color,
               linewidth=2, zorder=5, marker='s',
               label=f'ALADIN tau*={tau_star:.2f} mm/dia (caso CR2MET {tau_cr2:g} mm)')
    ax.plot([tau_cr2, tau_star], [f_cr2, f_cr2], linestyle=':', color=color, alpha=0.8)

ax.set_xlabel('Umbral de wet day tau (mm/dia)')
ax.set_ylabel('Fraccion integrada de wet days en Chile (%)')
ax.set_title(
    'Calibracion de umbral ALADIN vs CR2MET (dominio integrado)\n'
    f'{START_DATE[:4]}-{END_DATE[:4]} | grilla ALADIN | estilo JCLI-D-21-0590 sec. 3b',
    fontweight='bold',
)
ax.set_xlim(-0.05, 8.05)
ax.grid(True, alpha=0.3, linestyle=':')
ax.legend(loc='upper right', fontsize=9, frameon=True)
plt.tight_layout()
plt.show()

print('Metodo: F(tau) = promedio espacial sobre Chile de [% dias con pr >= tau].')
print('tau* en ALADIN resuelve F_ALADIN(tau*) = F_CR2MET(tau_ref) por biseccion.')